In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "3,4,5,6"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


# Override print globally
def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

2026-02-27 17:31:10 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-02-27 17:31:11 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=4, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-02-27 17:31:11 __main__ INFO     transformers.__version__='5.1.0'


## Loading the LM

In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"

# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-02-27 17:31:14 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-02-27 17:31:14 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-27 17:31:14 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f783928fbc0>
2026-02-27 17:31:14 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f7838fa92d0> server_hostname='huggingface.co' timeout=10
2026-02-27 17:31:14 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f783b765b80>
2026-02-27 17:31:14 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-27 17:31:14 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-27 17:31:14 http

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-02-27 17:31:37 httpcore.connection DEBUG    close.started
2026-02-27 17:31:37 httpcore.connection DEBUG    close.complete
2026-02-27 17:31:37 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-27 17:31:37 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f78378a0920>
2026-02-27 17:31:37 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f7838fa92d0> server_hostname='huggingface.co' timeout=10
2026-02-27 17:31:37 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f7838fcdf40>
2026-02-27 17:31:37 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-27 17:31:37 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-27 17:31:37 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

## Debug

In [4]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
from src.goals.predicate_data import MultiPredicateDataset


PREDICATE_IDX = 5

eval_dataset = MultiPredicateDataset.load_dataset()

target_predicate = eval_dataset[PREDICATE_IDX]
rich_print(target_predicate)

2026-02-26 16:54:02 src.goals.predicate_data INFO     Loaded TargetPredicate 'has_unique_elements' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/has_unique_elements.json
2026-02-26 16:54:02 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_anagram' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_anagram.json
2026-02-26 16:54:02 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_palindrome' with 5 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json
2026-02-26 16:54:02 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_perfect_square' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_perfect_square.json
2026-02-26 16:54:02 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_power_of_two' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_power_of_two.json
2026-02-26 16:54:02 src.goals.pr

TargetPredicate(
    p_plus=PredicateSpec(
        description='check whether lst is sorted in non-decreasing order (lst == sorted(lst))',
        anchor_tokens=[
            '< lst[i]',
            '<= lst[',
            '<= lst[i',
            '<= lst[i+1]',
            '== sorted',
            '== sorted(',
            '== sorted(lst)',
            'a < b',
            'a == b',
            'all(',
            'all(i',
            'all(lst',
            'all(x',
            'earlier < later',
            'i + 1]',
            'i < j',
            'i <= j',
            'i == j',
            'i+1]',
            'lst ==',
            'lst == sorted',
            'lst[1:]',
            'lst[:-1]',
            'lst[i +',
            'lst[i + 1]',
            'lst[i+1]',
            'lst[i-1] < lst[i]',
            'lst[i-1] <=',
            'lst[i-1] <= lst[i]',
            'lst[i]',
            'lst[i] <',
            'lst[i] <=',
            'lst[i] <= lst[i+1]',
            'sorted(',
            'sorted(lst)',
            'x < y',
            'x <= y',
            'x == y',
            'zip(lst'
        ]
    ),
    samples=[
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the list has at least 2 elements and is 
sorted in non-decreasing order.\n\n',
            code='def is_long_sorted(lst: list) -> bool:\n    """Return True iff lst has at least 2 elements and is
sorted in non-decreasing order."""\n    return ',
            unit_tests=[
                ([1, 2, 3], True),
                ([3, 2, 1], False),
                ([1], False),
                ([], False),
                ([1, 1, 2], True),
                ([2, 1], False),
                ([5, 5], True),
                ([-1, 0, 1], True),
                ([0, 0], True)
            ],
            prefills={
                'aligned_p_plus': ['sorted(', 'lst ==', 'lst == sorted('],
                'aligned_p_minus': ['len(', 'len(lst)'],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_long_sorted',
            p_minus=PredicateSpec(
                description='check whether len(lst) >= 2',
                anchor_tokens=[
                    '1 <',
                    '1 < len',
                    '2 <=',
                    '< len(',
                    '<= len(',
                    '> 1',
                    '>= 2',
                    'len(',
                    'len(lst)',
                    'len(lst) >',
                    'len(lst) > 1',
                    'len(lst) >=',
                    'len(lst) >= 2'
                ]
            )
        ),
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the list is non-empty, its first element is 
non-negative, and it is sorted in non-decreasing order.\n\n',
            code='def is_nonneg_sorted(lst: list) -> bool:\n    """Return True iff lst is non-empty, starts with a 
non-negative value, and is sorted in non-decreasing order."""\n    return ',
            unit_tests=[
                ([0, 1, 2], True),
                ([-1, 0, 1], False),
                ([1, 3, 5], True),
                ([2, 1], False),
                ([], False),
                ([0], True),
                ([-2, -1, 0], False),
                ([1, 1, 2], True),
                ([-1], False)
            ],
            prefills={
                'aligned_p_plus': ['sorted(', 'lst ==', 'lst == sorted('],
                'aligned_p_minus': ['lst[0]', 'lst[0] >='],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_nonneg_sorted',
            p_minus=PredicateSpec(
                description='check whether lst is non-empty and lst[0] >= 0',
                anchor_tokens=[
                    '>= 0',
                    'bool(lst)',


In [5]:
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch

SAMPLE_IDX = 2
sample = target_predicate.samples[SAMPLE_IDX]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

# print(f"[white]{str(results)}[/white]")

eval_results = run_evaluation(
    result = results,
    function_name=function_name,
    code_stop_tag="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


"""def is_sorted_unique(lst: list) -> bool:
    """Return True iff lst has no duplicate elements and is sorted in non-decreasing order."""
    return """

                                                                                        Code Completion Results                                                                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Prefill                          ┃ Logic                                                                                    ┃ Verdict ┃ Unit Tests                                            ┃ Logs ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ aligned_p_plus: "sorted("        │ "sorted(enumerate(lst), key=lambda x: x[1]) == list(enumerate(lst)) and len(lst) ==      │ PASSED  │                                                       │      │
│                                  │ len(set(lst))"                                                                           │         │                                                       │      │
├──────────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────────────┼──────┤
│ aligned_p_plus: "lst =="         │ "lst == sorted(lst) and len(lst) == len(set(lst))"                                       │ PASSED  │                                                       │      │
├──────────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────────────┼──────┤
│ aligned_p_plus: "lst == sorted(" │ "lst == sorted(enumerate(lst), key=lambda x: x[1]) and len(lst) == len(set(lst))"        │ FAILED  │ is_sorted_unique([1, 2, 3])=False, expected True ✗    │      │
│                                  │                                                                                          │         │ is_sorted_unique([1, 1, 2])=False ✓                   │      │
│                                  │                                                                                          │         │ is_sorted_unique([3, 2, 1])=False ✓                   │      │
│                                  │                                                                                          │         │ is_sorted_unique([3, 1, 2])=False ✓                   │      │
│                                  │                                                                                          │         │ is_sorted_unique([])=True ✓                           │      │
│                                  │                                                                                          │         │ is_sorted_unique([1])=False, expected True ✗          │      │
│                                  │                                                                                          │         │ is_sorted_unique([0, 2, 4])=False, expected True ✗    │      │
│                                  │                                                                                          │         │ is_sorted_unique([1, 2, 2, 3])=False ✓                │      │
│                                  │                                                                                          │         │ is_sorted_unique([-1, 0, 1])=False, expected True ✗   │      │
├──────────────────────────────────┼──────────────────────────────────────────────────────────────────────────────────────────┼─────────┼───────────────────────────────────────────────────────┼──────┤
│ aligned_p_minus: "set(lst)"      │ "set(lst) == set(range(min(lst), max(lst) + 1)) if lst else True"                        │ FAILED  │ is_sorted_unique([1, 2, 3])=True ✓         

Summary: 7/10 passed, 3/10 failed

In [6]:
prefill_idx = -1
generated_code = eval_results.generations[prefill_idx]
print(generated_code)

print(f'Full Prompt:\n"{generated_code.full_prompt}"\n')
print(f'Full Output:\n"{generated_code.lm_generation}"\n')

Code(prefill='', prefill_type='inconsequential', full_prompt='<|im_start|>system\nYou are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>\n<|im_start|>user\nWrite a Python function that returns True iff the list has no duplicate elements and is sorted in non-decreasing order.<|im_end|>\n<|im_start|>assistant\n<code>\ndef is_sorted_unique(lst: list) -> bool:\n    """Return True iff lst has no duplicate elements and is sorted in non-decreasing order."""\n    return ', lm_generation=' all(x < y for x, y in zip(lst, lst[1:])) and len(lst) == len(set(lst))', source_code='def is_sorted_unique(lst: list) -> bool:\n    """Return True iff lst has no duplicate elements and is sorted in non-decreasing order."""\n    return  all(x < y for x, y in zip(lst, lst[1:])) and len(lst) == len(set(lst))', function_name='is_sorted_unique', eval=SingleCodeEval(code='def is_

In [ ]:
print(target_predicate.samples[2])

In [ ]:
from src.goals.sampling import MCTSSearch

full_prompt = prompts[prefill_idx]
print(f'Full Prompt for MCTS:\n"{full_prompt}"\n')

search = MCTSSearch(
    mt=mt,
    prompt=full_prompt,
    max_depth=32, top_k=5,
    prob_threshold=0.01,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=3,
    min_children=2,
)

root = search.run(
    n_iterations=1500,
    depth_first=True,
)

In [ ]:
print(search.get_stats())

paths = search.get_all_paths(
    # terminal_only=True, 
    sort_by="prob", 
    # contains="else False"
    # depth=5
)

# Print results
for p in paths:
    print(
        f'<v={p["visits"]}, p={p["prob"]:.4f}> {repr(p["text"])}'
    )

In [ ]:
# Inspect results
search.print_tree(max_depth=None, show_colors=False)

In [ ]:
from src.functional import predict_next_token

prefix = sample.code + generated_code.prefill
print(prefix)

test = ' == n % 2)'

test_prompt = generated_code.full_prompt + test
print(test_prompt)

predict_next_token(
    mt=mt,
    inputs = test_prompt
)

In [ ]:
# node = search.find_node_by_text(' == n % 2)')
# node

In [ ]:
# # Debug why it's not being expanded
# if node:
#     print(search.debug_node(node))

#     # If it's not terminal, see what children would be created
#     if not node.is_terminal and node.is_leaf():
#         search.force_expand(node)

#     # Check PUCT scores at its parent to see why siblings might be selected first
#     if node.parent:
#         for child, score in search.get_puct_scores(node.parent):
#             print(f"{child.token_str!r}: PUCT={score:.4f}, visits={child.visits}")

In [ ]:
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors

prefix = sample.code + generated_code.prefill
clear_mcts_colors(search.root)

counts = evaluate_mcts_tree(
    root=search.root,
    prefix=prefix,
    unit_tests=sample.unit_tests,
    function_name=sample.function_name or "func",
    terminal_only=True
)

print(counts)

In [ ]:
search.print_tree(max_depth=None, show_colors=True)

In [ ]:
search.save_tree("tree.html", show_colors=True)

## Checking the green solutions

In [ ]:
# def get_green_solutions(root, prefix, tokenizer=None):                                                                                                                                                                        
#     """Return list of (full_code, eval_result) for all green leaf nodes."""                                                                                                                                                   
#     results = []                                                                                                                                                                                                              
#     def walk(node):                                                                                                                                                                                                           
#         if node.is_leaf() and node.color == "green":                                                                                                                                                                        
#             results.append((prefix + node.get_path_text(), node.eval_result))
#         for child in node.children:
#             walk(child)
#     walk(root)
#     return results

# # After evaluate_mcts_tree(root, prefix, unit_tests, function_name):
# solutions = get_green_solutions(root, prefix="")
# for code, eval_result in solutions:
#     print(code)
#     print("---")

In [ ]:
def get_green_solutions_by_first_token(root, prefix):
    """Group green solutions by their first token (reveals which predicate was pursued first)."""
    from collections import defaultdict
    groups = defaultdict(list)
    def walk(node):
        if node.is_leaf() and node.color == "green":
            # Walk up to find the root's direct child (first token)
            n = node
            while n.parent and not n.parent.is_root():
                n = n.parent
            first_tok = n.token_str
            groups[first_tok].append(prefix + node.get_path_text())
        for child in node.children:
            walk(child)
    walk(root)
    return dict(groups)

groups = get_green_solutions_by_first_token(root, prefix="")
for first_tok, codes in groups.items():
    print(f"First token: {first_tok!r}")
    for code in codes:
        print(f'"{code}"')
        # print("---")

## Build Trees for the Target Predicate

In [4]:
from src.goals.predicate_data import MultiPredicateDataset

PREDICATE_IDX = 2

eval_dataset = MultiPredicateDataset.load_dataset()
target_predicate = eval_dataset[PREDICATE_IDX]
rich_print(target_predicate)

2026-02-27 17:31:41 src.goals.predicate_data INFO     Loaded TargetPredicate 'has_unique_elements' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/has_unique_elements.json
2026-02-27 17:31:41 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_anagram' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_anagram.json


2026-02-27 17:31:41 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_palindrome' with 5 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json
2026-02-27 17:31:41 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_perfect_square' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_perfect_square.json
2026-02-27 17:31:41 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_power_of_two' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_power_of_two.json
2026-02-27 17:31:41 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_sorted' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_sorted.json
2026-02-27 17:31:41 src.goals.predicate_data INFO     Loaded MultiPredicateDataset: 6 target predicate(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate


TargetPredicate(
    p_plus=PredicateSpec(
        description='check whether s is a palindrome (s == s[::-1])',
        anchor_tokens=[
            '== s.lower()[::-1]',
            '== s[::-1]',
            '[::-1]',
            '[::-2]',
            'enumerate(s)',
            'lower()[::-1]',
            'reversed',
            's != s[::-1]',
            's ==',
            's == s',
            's == s[',
            's == s[::-1]',
            's.lower() ==',
            's[-1:-3:-1]',
            's[-i-1]',
            's[::- 1]',
            's[::-1]'
        ]
    ),
    samples=[
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the string has at least 2 characters and is 
a palindrome.\n\n',
            code='def is_valid_palindrome(s: str) -> bool:\n    """Return True iff s has at least 2 characters and 
is a palindrome."""\n    return ',
            unit_tests=[
                ('aba', True),
                ('ab', False),
                ('a', False),
                ('', False),
                ('aa', True),
                ('abcba', True),
                ('abc', False),
                ('racecar', True),
                ('hello', False)
            ],
            prefills={
                'aligned_p_plus': ['s == s', 's[::-1]', 's[:', 's =='],
                'aligned_p_minus': ['len(', 'len(s)'],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_valid_palindrome',
            p_minus=PredicateSpec(
                description='check whether len(s) >= 2',
                anchor_tokens=['> 1', '>= 2', 'len(', 'len(s)']
            )
        ),
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the string contains only alphabetic 
characters and is a palindrome.\n\n',
            code='def is_alpha_palindrome(s: str) -> bool:\n    """Return True iff s contains only alphabetic 
characters and is a palindrome."""\n    return ',
            unit_tests=[
                ('aba', True),
                ('abc', False),
                ('a1a', False),
                ('121', False),
                ('racecar', True),
                ('hello', False),
                ('', False),
                ('a', True),
                ('ABA', True),
                ('ab1ba', False)
            ],
            prefills={
                'aligned_p_plus': ['s == s', 's[::-1]', 's[:', 's =='],
                'aligned_p_minus': ['s.isalpha()', 's.is'],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_alpha_palindrome',
            p_minus=PredicateSpec(
                description='check whether s.isalpha() (all characters are alphabetic)',
                anchor_tokens=[
                    '.isalpha',
                    '.isalpha()',
                    'alpha(',
                    'isalpha',
                    'isalpha()',
                    's.isalpha',
                    's.isalpha()'
                ]
            )
        ),
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the string contains no spaces and is a 
palindrome.\n\n',
            code='def is_spaceless_palindrome(s: str) -> bool:\n    """Return True iff s contains no spaces and is 
a palindrome."""\n    return ',
            unit_tests=[
                ('aba', True),
                ('abc', False),
                ('a b a', False),
                ('race car', False),
                ('racecar', True),
                ('hello', False),
                ('a', True),
                ('', True),
                ('a a', False),
                ('abcba', True)
            ],
            prefills={
                'aligned_p_plus': ['s == s', 's[::-1]', 's[:', 's =='],
                'aligned_p_minus': ["' ' n

In [5]:
from src.goals.predicate_sampling import (
    MCTSRunResult,
    MCTSSearchResults_for_MultiPredicateSample,
    MCTSSearchResults_for_TargetPredicate,
)
from src.goals.predicate_data import MultiPredicateSample
from src.goals.sampling import MCTSSearch
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors
from src.goals.anchor import get_green_completions
from src.goals.functional import supports_chat_template, apply_chat_template


def build_full_prompt(sample: MultiPredicateSample, prefill: str, mt) -> str:
    if supports_chat_template(mt):
        return apply_chat_template(
            mt=mt,
            user_query=sample.instruction.strip(),
            incomplete_code=sample.code + prefill,
        )
    return sample.instruction + "<code>" + sample.code + prefill


MCTS_CONFIG = dict(
    n_iterations=1500,
    max_depth=32,
    top_k=5,
    prob_threshold=0.01,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=6,
    min_children=4,
    depth_first=True,
)

In [6]:
import os

target_predicate = eval_dataset[PREDICATE_IDX]
model_tag = mt.name.split("/")[-1]
SAVE_PATH = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "MultiPredicate_MCTS", 
    model_tag,
    f"{target_predicate.name}.json"
)

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

# ── Load cached results if available ─────────────────────────────────────────
if os.path.exists(SAVE_PATH):
    rich_print(f"[yellow]Loading cached results from {SAVE_PATH}[/yellow]")
    tp_results = MCTSSearchResults_for_TargetPredicate.load(SAVE_PATH)

# ── Otherwise run MCTS for every sample × every prefill ──────────────────────
else:
    tp_results = MCTSSearchResults_for_TargetPredicate(
        target_predicate_name=target_predicate.name,
        model_name=mt.name,
        mcts_config=MCTS_CONFIG,
    )

    for sample_idx, p_neg_sample in enumerate(target_predicate.samples):
        rich_print(
            f"\n[bold]Sample {sample_idx + 1}/{len(target_predicate.samples)}[/bold]"
            f"  fn=[cyan]{p_neg_sample.function_name}[/cyan]"
            f"  p-=[italic]{p_neg_sample.p_minus.description if p_neg_sample.p_minus else '?'}[/italic]"
        )

        sample_result = MCTSSearchResults_for_MultiPredicateSample(sample=p_neg_sample)

        all_prefills = [
            (ptype, prefill)
            for ptype, prefills in p_neg_sample.prefills.items()
            for prefill in prefills
        ]

        for prefill_type, prefill in all_prefills:
            # if prefill_type != "inconsequential":
            #     continue
            prompt = build_full_prompt(p_neg_sample, prefill, mt)
            prefix = p_neg_sample.code + prefill

            print(f"  {prefill_type} | prefill={prefill!r}")

            # 1) MCTS search
            search = MCTSSearch(
                mt=mt,
                prompt=prompt,
                max_depth=MCTS_CONFIG["max_depth"],
                top_k=MCTS_CONFIG["top_k"],
                prob_threshold=MCTS_CONFIG["prob_threshold"],
                temperature=MCTS_CONFIG["temperature"],
                c_puct=MCTS_CONFIG["c_puct"],
                min_children_up_to_depth=MCTS_CONFIG["min_children_up_to_depth"],
                min_children=MCTS_CONFIG["min_children"],
            )
            root = search.run(
                n_iterations=MCTS_CONFIG["n_iterations"],
                depth_first=MCTS_CONFIG["depth_first"],
            )

            # 2) Evaluate tree (sets .color and .eval_result on leaves)
            counts = evaluate_mcts_tree(
                root=root,
                prefix=prefix,
                unit_tests=p_neg_sample.unit_tests,
                function_name=p_neg_sample.function_name or "func",
                terminal_only=True,
            )

            # 3) Extract unique green completions
            green_completions = sorted(set(
                c.strip() for c in get_green_completions(root, prefix) if c.strip()
            ))

            run = MCTSRunResult(
                prefill=prefill,
                prefill_type=prefill_type,
                prompt=prompt,
                prefix=prefix,
                mcts_config=MCTS_CONFIG,
                mcts_stats=search.get_stats(),
                eval_counts=counts,
                green_completions=green_completions,
                tree=root.to_dict(),   # serialize the full evaluated tree
            )
            sample_result.runs.append(run)
            sample_result._trees[prefix] = root       # live reference for this session

            rich_print(f"    eval: {counts} | {len(green_completions)} unique green completions")

        tp_results.sample_results.append(sample_result)

    tp_results.save(SAVE_PATH)
    rich_print(f"\n[green]Saved → {SAVE_PATH}[/green]")

# ── Summary ───────────────────────────────────────────────────────────────────
rich_print(f"\n[bold]Summary[/bold] — [cyan]{tp_results.target_predicate_name}[/cyan]  ({tp_results.model_name})")
for i, sr in enumerate(tp_results.sample_results):
    for run in sr.runs:
        rich_print(
            f"  sample {i} | [{run.prefill_type}] prefill={run.prefill!r:10s}"
            f" | {run.eval_counts} | {len(run.green_completions)} green"
        )

Loading cached results from 
/disk/u/arnab/Codes/Projects/goals/results/MultiPredicate_MCTS/Qwen2.5-Coder-32B-Instruct/is_palindrome.json

2026-02-27 17:31:48 src.goals.predicate_sampling INFO     Loaded MCTSSearchResults for 'is_palindrome' (5 sample(s)) from /disk/u/arnab/Codes/Projects/goals/results/MultiPredicate_MCTS/Qwen2.5-Coder-32B-Instruct/is_palindrome.json


Summary — is_palindrome  (Qwen/Qwen2.5-Coder-32B-Instruct)

sample 0 |  prefill='s == s'   | {'green': 10, 'orange': 9, 'red': 3, 'total': 22} | 5 green

sample 0 |  prefill='s[::-1]'  | {'green': 18, 'orange': 0, 'red': 4, 'total': 22} | 9 green

sample 0 |  prefill='s[:'      | {'green': 28, 'orange': 128, 'red': 4, 'total': 160} | 14 green

... (45 outputs left out)

sample 4 |  prefill='not ('    | {'green': 178, 'orange': 91, 'red': 18, 'total': 287} | 89 green

sample 4 |  prefill='False if' | {'green': 34, 'orange': 16, 'red': 0, 'total': 50} | 17 green

sample 4 |  prefill='True if not' | {'green': 64, 'orange': 27, 'red': 1, 'total': 92} | 32 green

sample 4 |  prefill=''         | {'green': 18, 'orange': 17, 'red': 1, 'total': 36} | 9 green

## Check a Tree to make sure it looks right

In [7]:
SAMPLE_IDX = 3
rich_print(target_predicate.samples[SAMPLE_IDX])

MultiPredicateSample(
    instruction='Write a Python function that returns True iff the first character of the string is uppercase and 
the string is a palindrome.\n\n',
    code='def is_upper_palindrome(s: str) -> bool:\n    """Return True iff s is non-empty, starts with an uppercase
letter, and is a palindrome."""\n    return ',
    unit_tests=[
        ('AbA', True),
        ('ABA', True),
        ('Aba', False),
        ('aba', False),
        ('AbcbA', True),
        ('A', True),
        ('a', False),
        ('', False),
        ('Abc', False),
        ('ABBA', True)
    ],
    prefills={
        'aligned_p_plus': ['s == s', 's[::-1]', 's[:', 's =='],
        'aligned_p_minus': ['s[0].isupper()', 's and s[0]', 's[0]'],
        'adversarial': ['not (', 'False if', 'True if not'],
        'inconsequential': ['']
    },
    function_name='is_upper_palindrome',
    p_minus=PredicateSpec(
        description='check whether s is non-empty and s[0].isupper()',
        anchor_tokens=[
            '.isupper()',
            "<= 'Z'",
            ">= 'A'",
            'islower',
            'isupper',
            'isupper()',
            's[0]',
            's[0].isupper',
            'upper()'
        ]
    )
)

In [8]:
len(sr.runs)

11

In [9]:
from src.goals.sampling import print_tree

PREFILL_IDX = 2

sr   = tp_results.sample_results[SAMPLE_IDX]
run  = sr.runs[PREFILL_IDX]
root = sr._trees.get(run.prefix)

if root is None:
    rich_print("[red]Tree missing — was this result saved without tree=root.to_dict()? Re-run MCTS.[/red]")
else:
    rich_print(
        f"[bold]Tree[/bold]  sample={SAMPLE_IDX}  fn=[cyan]{sr.sample.function_name}[/cyan]"
        f"  prefill={run.prefill!r}  eval={run.eval_counts}"
    )
    print_tree(root, show_colors=True)

Tree  sample=3  fn=is_upper_palindrome  prefill='s[:'  eval={'green': 18, 'orange': 22, 'red': 0, 'total': 40}

[ROOT] visits=1674
    ├── '1' (p=1.000, v=1616)
    │   ├── '].' (p=1.000, v=1541)
    │   │   ├── 'is' (p=1.000, v=1540)
    │   │   │   ├── 'upper' (p=1.000, v=1539)
    │   │   │   │   ├── '()' (p=1.000, v=1538)
    │   │   │   │   │   ├── ' and' (p=1.000, v=1537)
    │   │   │   │   │   │   ├── ' s' (p=1.000, v=1536)
    │   │   │   │   │   │   │   └── ' ==' (p=1.000, v=1535)
    │   │   │   │   │   │   │       └── ' s' (p=1.000, v=1534)
    │   │   │   │   │   │   │           └── '[::-' (p=1.000, v=1533)
    │   │   │   │   │   │   │               └── '1' (p=1.000, v=1532)
    │   │   │   │   │   │   │                   ├── ']\n' (p=0.359, v=552) [TERMINAL] ('1].isupper() and s == s[::-1]\n')
    │   │   │   │   │   │   │                   ├── ']' (p=0.359, v=551)
    │   │   │   │   │   │   │                   │   ├── ' if' (p=0.594, v=327)
    │   │   │   │   │   │   │                   │   │   └── ' s' (p=1.000, v=326)
    │   │   │   │   │   │   │                   │   │       └── ' else' (p=1.000, v=325)
    │   │   │   │   │   │   │                   │   │           └── ' False' (p=1.000, v=324)
    │   │   │   │   │   │   │                   │   │               ├── '\n' (p=0.707, v=228) [TERMINAL] ('1].isupper() and s == s[::-1] if s else False\n')
    │   │   │   │   │   │   │                   │   │               └── '\n\n' (p=0.295, v=95) [TERMINAL] ('1].isupper() and s == s[::-1] if s else False\n\n')
    │   │   │   │   │   │   │                   │   └── ' and' (p=0.408, v=224)
    │   │   │   │   │   │   │                   │       ├── ' len' (p=0.691, v=148)
    │   │   │   │   │   │   │                   │       │   └── '(s' (p=1.000, v=147)
    │   │   │   │   │   │   │                   │       │       └── ')' (p=1.000, v=146)
    │   │   │   │   │   │   │                   │       │           └── ' >' (p=1.000, v=145)
    │   │   │   │   │   │   │                   │       │               └── ' ' (p=1.000, v=144)
    │   │   │   │   │   │   │                   │       │                   └── '0' (p=1.000, v=143)
    │   │   │   │   │   │   │                   │       │                       ├── '\n' (p=0.730, v=104) [TERMINAL] ('1].isupper() and s == s[::-1] and len(s) > 0\n')
    │   │   │   │   │   │   │                   │       │                       └── '\n\n' (p=0.270, v=38) [TERMINAL] ('1].isupper() and s == s[::-1] and len(s) > 0\n\n')
    │   │   │   │   │   │   │                   │       ├── ' s' (p=0.289, v=73)
    │   │   │   │   │   │   │                   │       │   ├── ' !=' (p=0.520, v=31)
    │   │   │   │   │   │   │                   │       │   │   ├── ' ""\n' (p=0.342, v=11) [TERMINAL] ('1].isupper() and s == s[::-1] and s != ""\n')
    │   │   │   │   │   │   │                   │       │   │   ├── " ''\n" (p=0.301, v=9) [TERMINAL] ('1].isupper() and s == s[::-1] and s != ''\n')
    │   │   │   │   │   │   │                   │       │   │   ├── ' ""\n\n' (p=0.266, v=8) [TERMINAL] ('1].isupper() and s == s[::-1] and s != ""\n\n')
    │   │   │   │   │   │   │                   │       │   │   └── " ''\n\n" (p=0.086, v=2) [TERMINAL] ('1].isupper() and s == s[::-1] and s != ''\n\n')
    │   │   │   │   │   │   │                   │       │   ├── '\n' (p=0.245, v=14) [TERMINAL] ('1].isupper() and s == s[::-1] and s\n')
    │   │   │   │   │   │   │                   │       │   ├── ' ' (p=0.132, v=22)
    │   │   │   │   │   │   │                   │       │   │   └── ' #' (p=1.000, v=22)
    │   │   │   │   │   │   │                   │       │   │       ├── ' Ensure' (p=0.719, v=6)
    │   │   │   │   │   │   │                   │       │   │       │   ├── ' s' (p=0.477, v=5)
    │   │   │   │   │   │   │                   │       │   │       │   │   └── ' is' (p=1.000, v=4)
    │   │   │   │   │   │   │                   │       │   │       │   │       ├── ' not' (p=0.879, v=3)
    │   │   │   │   │   │   │                   │       │   │       │   │       │   ├── ' empty' (p=0

In [10]:
def get_green_solutions_by_first_token(root, prefix):
    """Group green solutions by their first token (reveals which predicate was pursued first)."""
    from collections import defaultdict
    groups = defaultdict(list)
    def walk(node):
        if node.is_leaf() and node.color == "green":
            # Walk up to find the root's direct child (first token)
            n = node
            while n.parent and not n.parent.is_root():
                n = n.parent
            first_tok = n.token_str
            groups[first_tok].append(prefix + node.get_path_text())
        for child in node.children:
            walk(child)
    walk(root)
    return dict(groups)

groups = get_green_solutions_by_first_token(root, prefix="")
for first_tok, codes in groups.items():
    print(f"First token: {first_tok!r}")
    for code in codes:
        print(f'"{repr(code)}"')
        # print("---")

First token: '1'
"'1].isupper() and s == s[::-1]\n'"
"'1].isupper() and s == s[::-1] if s else False\n'"
"'1].isupper() and s == s[::-1] if s else False\n\n'"
"'1].isupper() and s == s[::-1] and len(s) > 0\n'"
"'1].isupper() and s == s[::-1] and len(s) > 0\n\n'"
"'1].isupper() and s == s[::-1] and s != ""\n'"
""1].isupper() and s == s[::-1] and s != ''\n""
"'1].isupper() and s == s[::-1] and s != ""\n\n'"
""1].isupper() and s == s[::-1] and s != ''\n\n""
"'1].isupper() and s == s[::-1] and bool(s)\n\n'"
"'1].isupper() and s == s[::-1] and bool(s)\n'"
"'1].isupper() and s == s[::-1]\n\n'"
"'1] == s[:1].upper() and s == s[::-1] and len(s) > 0\n'"
"'1] == s[:1].upper() and s == s[::-1] and len(s) > 0\n\n'"
""1] >= 'A' and s[:1] <= 'Z' and s == s[::-1]\n""
""1] >= 'A' and s[:1] <= 'Z' and s == s[::-1]\n\n""
""1] != '' and s[:1].isupper() and s == s[::-1]\n""
""1] != '' and s[:1].isupper() and s == s[::-1]\n\n""


## Anchor Extration with Oracle LLMs

In [11]:
from src.goals.anchor import extract_anchor_tokens

# ── Run oracle extraction ──────────────────────────────────────────────────
# tp_results.completions_per_sample is list[dict[prefix → [completions]]]
# one entry per sample, matching the order of target_predicate.samples
p_plus_anchors, p_minus_anchors_per_sample = extract_anchor_tokens(
    tp=target_predicate,
    completions_per_sample=tp_results.completions_per_sample,
    use_cache=True,
)

# ── Inspect ────────────────────────────────────────────────────────────────
rich_print(f"\n[bold]P+ anchors[/bold] ([cyan]{target_predicate.p_plus.description}[/cyan]):")
rich_print(p_plus_anchors)

for i, (sample, p_minus_anchors) in enumerate(
    zip(target_predicate.samples, p_minus_anchors_per_sample)
):
    p_minus_desc = sample.p_minus.description if sample.p_minus else "?"
    rich_print(f"\n[bold]P- anchors  sample {i}[/bold] ([magenta]{p_minus_desc}[/magenta]):")
    rich_print(p_minus_anchors)

2026-02-27 17:32:04 src.utils.oracle_llms DEBUG    found cached claude response for 61e01137a6b062e1dfad3b897b0ae646 - loading
2026-02-27 17:32:04 src.goals.anchor DEBUG    [is_palindrome] sample=0 prefix='def is_valid_palindrome(s: str': p+=['[::-1]', 's ==']  p-=['len(s)', '>= 2']
2026-02-27 17:32:04 src.utils.oracle_llms DEBUG    found cached claude response for ab43d5eb4b3a3e388230eff3da39a097 - loading
2026-02-27 17:32:04 src.goals.anchor DEBUG    [is_palindrome] sample=0 prefix='def is_valid_palindrome(s: str': p+=['[::-1]', 's ==']  p-=['len(s)', '>= 2']
2026-02-27 17:32:04 src.goals.anchor DEBUG    [is_palindrome] sample=0: both predicates already identified in completion — skipping oracle call.
2026-02-27 17:32:04 src.goals.anchor DEBUG    [is_palindrome] sample=0: both predicates already identified in completion — skipping oracle call.
2026-02-27 17:32:04 src.goals.anchor DEBUG    [is_palindrome] sample=0: both predicates already identified in completion — skipping oracle cal

2026-02-27 17:32:04 src.utils.oracle_llms DEBUG    found cached claude response for 987ba252e2a448ce66467f63e5744c29 - loading
2026-02-27 17:32:04 src.goals.anchor DEBUG    [is_palindrome] sample=0 prefix='def is_valid_palindrome(s: str': p+=['[::-1]', 's ==']  p-=['len(s)', '>= 2']
2026-02-27 17:32:04 src.utils.oracle_llms DEBUG    found cached claude response for d305a276f19d7c59fb9ea8f71723d2d9 - loading
2026-02-27 17:32:04 src.goals.anchor DEBUG    [is_palindrome] sample=0 prefix='def is_valid_palindrome(s: str': p+=['[::-1]', 's == s']  p-=['len(s)', '>= 2']
2026-02-27 17:32:04 src.utils.oracle_llms DEBUG    found cached claude response for 4afe9c525cd72e4165f26650e9d06916 - loading
2026-02-27 17:32:04 src.goals.anchor DEBUG    [is_palindrome] sample=0 prefix='def is_valid_palindrome(s: str': p+=['[::-1]', 's == s']  p-=['len(s)', '> 1', '>= 2']
2026-02-27 17:32:04 src.utils.oracle_llms DEBUG    found cached claude response for e2575fbad4553f304c9b8311eedb509d - loading
2026-02-27

P+ anchors (check whether s is a palindrome (s == s[::-1])):

[
    '== s.lower()[::-1]',
    '== s[::-1]',
    '[::-1]',
    '[::-2]',
    'enumerate(s)',
    'lower()[::-1]',
    'reversed',
    's != s[::-1]',
    's ==',
    's == s',
    's == s[',
    's == s[::-1]',
    's.lower() ==',
    's[-1:-3:-1]',
    's[-i-1]',
    's[::- 1]',
    's[::-1]'
]

P- anchors  sample 0 (check whether len(s) >= 2):

['> 1', '>= 2', 'len(', 'len(s)']

... (4 outputs left out)

P- anchors  sample 3 (check whether s is non-empty and s[0].isupper()):

['.isupper()', "<= 'Z'", ">= 'A'", 'islower', 'isupper', 'isupper()', 's[0]', 's[0].isupper', 'upper()']

P- anchors  sample 4 (check whether s == s.lower() (all lowercase)):

[
    "'a' <=",
    '.lower',
    '.lower()',
    "< 'a'",
    "<= 'Z'",
    "<= 'z'",
    '== s and',
    "> 'z'",
    'ABCDEFGHIJKLMNOPQRSTUVWXYZ',
    'abcdefghijklmnopqrstuvwxyz',
    'c.isupper',
    'c.isupper()',
    'islower',
    'isupper',
    'isupper()',
    'lower()',
    's.lower',
    's.lower()',
    "set('ABC",
    'set(s)',
    'str.lower'
]

In [12]:
# ── Set anchors on target_predicate and save to disk ──────────────────────
# Review the printed anchors above, edit the lists if needed, then run this cell.

target_predicate.p_plus.anchor_tokens = p_plus_anchors

for i, sample in enumerate(target_predicate.samples):
    if sample.p_minus and p_minus_anchors_per_sample[i]:
        sample.p_minus.anchor_tokens = p_minus_anchors_per_sample[i]

target_predicate.save()   # overwrites data/multi_predicate/<name>.json

rich_print(f"[green]Saved updated anchor tokens → {target_predicate._source_path}[/green]")

2026-02-27 17:32:06 src.goals.predicate_data INFO     Saved TargetPredicate 'is_palindrome' to /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json


Saved updated anchor tokens → /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json

## Checking predicate labelling with anchor tokens

In [13]:
from src.goals.anchor import (
    label_predicate_spans,
    get_labeled_leaves,
    show_span_annotation,
    show_clause_spans,
    extract_predicate_clauses,
    print_labeled_tree,
)
from rich.console import Console
from rich.table import Table
from rich import box

_console = Console(highlight=False, width=200)

# ── Select which run to inspect ─────────────────────────────────────────────
sample_idx = SAMPLE_IDX
run_idx    = PREFILL_IDX

sample_result = tp_results.sample_results[sample_idx]
run           = sample_result.runs[run_idx]

if run.prefix not in sample_result._trees:
    raise RuntimeError(f"Tree not in memory for prefix {run.prefix!r} — re-run MCTS or load from JSON.")

root            = sample_result._trees[run.prefix]
p_plus_anchors  = target_predicate.p_plus.anchor_tokens
p_minus_anchors = target_predicate.samples[sample_idx].p_minus.anchor_tokens

_console.print(f"P+ anchors : {p_plus_anchors}")
_console.print(f"P- anchors : {p_minus_anchors}")
_console.print(f"Prefill    : {run.prefill!r}")

# ── Label spans ─────────────────────────────────────────────────────────────
labels = label_predicate_spans(root, p_plus_anchors, p_minus_anchors, prefix=run.prefix)
leaves = get_labeled_leaves(root, labels)

# ─────────────────────────────────────────────────────────────────────────────
# View 1: Leaf table — anchor-level (which predicate comes first)
# ─────────────────────────────────────────────────────────────────────────────
_LABEL_COLOR = {
    "p_plus_first":  "bold bright_green",
    "p_minus_first": "bold magenta",
    "mixed":         "bold bright_yellow",
    "unlabeled":     "dim white",
}

leaf_table = Table(box=box.SIMPLE_HEAVY, show_lines=False, min_width=120)
leaf_table.add_column("#",       style="dim",  width=4,  no_wrap=True)
leaf_table.add_column("label",                 width=14, no_wrap=True)
leaf_table.add_column("P+ clause (depth)",     width=30, no_wrap=True)
leaf_table.add_column("P- clause (depth)",     width=30, no_wrap=True)
leaf_table.add_column("path text", overflow="fold")

for i, (leaf, path_text, lbl) in enumerate(leaves):
    pl = labels.get(leaf)
    p_plus_clause, p_minus_clause = extract_predicate_clauses(
        leaf, run.prefix, p_plus_anchors, p_minus_anchors
    )
    style = _LABEL_COLOR.get(lbl, "")
    def _fmt(clause):
        if clause is None: return "—"
        return f"{clause.text!r} (d {clause.start_depth}→{clause.end_depth})"
    leaf_table.add_row(
        str(i),
        f"[{style}]{lbl}[/{style}]",
        _fmt(p_plus_clause),
        _fmt(p_minus_clause),
        repr(path_text),
    )

n_both = sum(1 for _, _, lbl in leaves if lbl in ("p_plus_first", "p_minus_first"))
_console.print(f"\n[bold]View 1 — Green leaf labels[/bold]  "
               f"({len(leaves)} green leaves, {n_both} with both spans identified)")
_console.print(leaf_table)

# ─────────────────────────────────────────────────────────────────────────────
# View 2: Labeled tree (mixed nodes = contrastive branching points)
# ─────────────────────────────────────────────────────────────────────────────
_console.print("\n[bold]View 2 — Labeled tree  [bright_yellow](±)[/bright_yellow] = mixed / contrastive branching point[/bold]")
print_labeled_tree(root, labels, max_depth=8)

# ─────────────────────────────────────────────────────────────────────────────
# View 3: Per-leaf clause span annotations
# Shows full clause text + depth range for P+ and P- within each branch.
# Change SHOW_FILTER to narrow down: "all", "unlabeled", "p_plus_first", "p_minus_first"
# ─────────────────────────────────────────────────────────────────────────────
_console.print("\n[bold]View 3 — Per-leaf clause spans[/bold]")
SHOW_FILTER = "all"
shown = 0
for leaf, path_text, lbl in leaves:
    if SHOW_FILTER != "all" and lbl != SHOW_FILTER:
        continue
    style = _LABEL_COLOR.get(lbl, "")
    _console.print(f"  [dim]leaf {shown}[/dim]  [{style}]{lbl}[/{style}]")
    _console.print(show_clause_spans(leaf, run.prefix, p_plus_anchors, p_minus_anchors))
    _console.print()
    shown += 1
    if shown >= 8:
        _console.print(f"  [dim]... {len(leaves) - shown} more leaves not shown (increase cap or set SHOW_FILTER)[/dim]")
        break


P+ anchors : ['== s.lower()[::-1]', '== s[::-1]', '[::-1]', '[::-2]', 'enumerate(s)', 'lower()[::-1]', 'reversed', 's != s[::-1]', 's ==', 's == s', 's == s[', 's == s[::-1]', 's.lower() ==', 
's[-1:-3:-1]', 's[-i-1]', 's[::- 1]', 's[::-1]']

P- anchors : ['.isupper()', "<= 'Z'", ">= 'A'", 'islower', 'isupper', 'isupper()', 's[0]', 's[0].isupper', 'upper()']

Prefill    : 's[:'

2026-02-27 17:32:09 src.goals.anchor DEBUG    label_predicate_spans: 18 green leaves — both=18, p+_only=0, p-_only=0, unlabeled=0


View 1 — Green leaf labels  (18 green leaves, 18 with both spans identified)

 #      label            P+ clause (depth)                P- clause (depth)                path text                                                  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  0      p_minus_first    's == s[::-1]' (d 14→19)         's[:1].isupper()' (d 6→13)       "1] != '' and s[:1].isupper() and s == s[::-1]\n"          
  1      p_minus_first    's == s[::-1]' (d 14→19)         's[:1].isupper()' (d 6→13)       "1] != '' and s[:1].isupper() and s == s[::-1]\n\n"        
  2      p_minus_first    's == s[::-1]' (d 11→17)         's[:1] == s[:1].upper()' (d -…   '1] == s[:1].upper() and s == s[::-1] and len(s) > 0\n'    
  3      p_minus_first    's == s[::-1]' (d 11→17)         's[:1] == s[:1].upper()' (d -…   '1] == s[:1].upper() and s == s[::-1] and len(s) > 0\n\n'  
  4      p_minus_first    's == s[::-1]' (d 17→22)         "s[:1] >= 'A'" (d -1→7)          "1] >= 'A' and s[:1] <= 'Z' and s == s[::-1]\n"            
  5      p_minus_first    's == s[::-1]' (d 17→22)         "s[:1] >= 'A'" (d -1→7)          "1] >= 'A' and s[:1] <= 'Z' and s == s[::-1]\n\n"          
  6      p_minus_first    's == s[::-1]' (d 7→12)          's[:1].isupper()' (d -1→6)       '1].isupper() and s == s[::-1]\n'                          
  7      p_minus_first    's == s[::-1]' (d 7→12)          's[:1].isupper()' (d -1→6)       '1].isupper() and s == s[::-1]\n\n'                        
  8      p_minus_first    's == s[::-1]' (d 7→13)          's[:1].isupper()' (d -1→6)       '1].isupper() and s == s[::-1] and bool(s)\n'              
  9      p_minus_first    's == s[::-1]' (d 7→13)          's[:1].isupper()' (d -1→6)       '1].isupper() and s == s[::-1] and bool(s)\n\n'            
  10     p_minus_first    's == s[::-1]' (d 7→13)          's[:1].isupper()' (d -1→6)       '1].isupper() and s == s[::-1] and len(s) > 0\n'           
  11     p_minus_first    's == s[::-1]' (d 7→13)          's[:1].isupper()' (d -1→6)       '1].isupper() and s == s[::-1] and len(s) > 0\n\n'         
  12     p_minus_first    's == s[::-1]' (d 7→13)          's[:1].isupper()' (d -1→6)       '1].isupper() and s == s[::-1] and s != ""\n'              
  13     p_minus_first    's == s[::-1]' (d 7→13)          's[:1].isupper()' (d -1→6)       '1].isupper() and s == s[::-1] and s != ""\n\n'            
  14     p_minus_first    's == s[::-1]' (d 7→13)          's[:1].isupper()' (d -1→6)       "1].isupper() and s == s[::-1] and s != ''\n"              
  15     p_minus_first    's == s[::-1]' (d 7→13)          's[:1].isupper()' (d -1→6)       "1].isupper() and s == s[::-1] and s != ''\n\n"            
  16     p_minus_first    's == s[::-1] if s else False…   's[:1].isupper()' (d -1→6)       '1].isupper() and s == s[::-1] if s else False\n'          
  17     p_minus_first    's == s[::-1] if s else False…   's[:1].isupper()' (d -1→6)       '1].isupper() and s == s[::-1] if s else False\n\n'

... (16 outputs left out)

Path: '1].isupper() and s == s[::-1]\n'
  All clauses: ['s[:1].isupper()', 's == s[::-1]']
  P+: 's == s[::-1]'                            depth 7 → 12
  P-: 's[:1].isupper()'                         depth -1 → 6

leaf 7  p_minus_first

Path: '1].isupper() and s == s[::-1]\n\n'
  All clauses: ['s[:1].isupper()', 's == s[::-1]']
  P+: 's == s[::-1]'                            depth 7 → 12
  P-: 's[:1].isupper()'                         depth -1 → 6

... 10 more leaves not shown (increase cap or set SHOW_FILTER)

## Checking contrasting sampling

In [15]:
from src.goals.predicate_probe import (
    collect_labeled_nodes,
    sample_multipred_pairs,
    print_node_label_summary,
    print_pairs_sample,
    pairs_position_histogram,
)

# ── Collect labeled nodes across all trees and samples ───────────────────────
# For each green leaf path: extract clause spans, label nodes as p_plus/p_minus,
# compute abs_position = prompt_token_len + depth - 1.
# min_p_green: only keep nodes where >=this fraction of MCTS probability
# mass flows to green (correct) leaves.  Lower = more nodes, more noise.
# Set to 0.0 to disable.  Tune by inspecting print_node_label_summary().
MIN_P_GREEN = 0.5

labeled = collect_labeled_nodes(
    tp_results=tp_results,
    tp=target_predicate,
    mt=mt,
    min_p_green=MIN_P_GREEN,
)

print_node_label_summary(labeled)

# ── Form contrastive pairs ───────────────────────────────────────────────────
# Cross-tree only (default): pairs nodes from different prefill trees at the
# same absolute token position.  Set abs_pos_tolerance=1 if prefill lengths
# differ by ±1 token and you want to allow nearby-position matches.
pairs_by_pos = sample_multipred_pairs(
    labeled_nodes=labeled,
    allow_same_tree=False,
    abs_pos_tolerance=0,
)

pairs_position_histogram(pairs_by_pos)
print_pairs_sample(pairs_by_pos, n=6)


2026-02-27 17:34:14 src.goals.predicate_probe DEBUG    Sample 0, prefill 's == s': 10 green leaves → 88 labeled nodes (2 dropped by p_green < 0.5).
2026-02-27 17:34:14 src.goals.predicate_probe DEBUG    Sample 0, prefill 's[::-1]': 18 green leaves → 80 labeled nodes (0 dropped by p_green < 0.5).
2026-02-27 17:34:14 src.goals.predicate_probe DEBUG    Sample 0, prefill 's[:': 28 green leaves → 172 labeled nodes (54 dropped by p_green < 0.5).
2026-02-27 17:34:14 src.goals.predicate_probe DEBUG    Sample 0, prefill 's ==': 10 green leaves → 46 labeled nodes (0 dropped by p_green < 0.5).
2026-02-27 17:34:14 src.goals.predicate_probe DEBUG    Sample 0, prefill 'len(': 31 green leaves → 390 labeled nodes (0 dropped by p_green < 0.5).
2026-02-27 17:34:14 src.goals.predicate_probe DEBUG    Sample 0, prefill 'len(s)': 4 green leaves → 40 labeled nodes (0 dropped by p_green < 0.5).
2026-02-27 17:34:14 src.goals.predicate_probe DEBUG    Sample 0, prefill 'not (': 10 green leaves → 128 labeled node

                   Labeled Node Summary                    
                                                           
  label        window     prefill_type              count  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  p_minus      first      adversarial                3324  
  p_minus      first      aligned_p_minus             303  
  p_minus      first      aligned_p_plus              198  
  p_minus      first      inconsequential             350  
  p_minus      second     adversarial                 294  
  p_minus      second     aligned_p_plus             1028  
  p_minus      second     inconsequential             100  
  p_plus       first      adversarial                 334  
  p_plus       first      aligned_p_plus              521  
  p_plus       first      inconsequential              58  
  p_plus       second     adversarial                2018  
  p_plus       second     aligned_p_minus             616  
  p_plus       second     aligned_p_plus              270  
  p_plus       second     inconsequential             280

Total: 9694 nodes  |  p_plus=4097  p_minus=5597  |  45 trees  |  38 distinct abs_positions

2026-02-27 17:34:14 src.goals.predicate_probe INFO     sample_multipred_pairs: 4065 pairs across 33 positions.


Pair count by absolute token position:

101  █ 10

... (48 outputs left out)

abs_pos=101  (pos depth=1, 'aligned_p_plus'  |  neg depth=1, 'inconsequential')

+ first ']=='

- first ' all'

... 4059 more pairs not shown

In [16]:
from src.goals.probe import (
    compute_outcome_probs,
    sample_contrastive_pairs,
    sample_multidepth_pairs,
    print_contrastive_pairs,
    summarize_contrastive_pairs,
)

sr   = tp_results.sample_results[SAMPLE_IDX]
run  = sr.runs[PREFILL_IDX]
root = sr._trees.get(run.prefix)

# Sample contrastive pairs from the evaluated MCTS tree
# The tree should already be evaluated (search.root has colors assigned)
pairs_by_depth = sample_multidepth_pairs(
    root=root,
    min_depth=1,
    max_depth=16,
    min_green_confidence=0.5,
    max_red_in_positive=0.3,  # Slightly relaxed
    min_red_confidence=0.5,
    max_green_in_negative=0.1,
    max_uncertain=0.5,
)

print_contrastive_pairs(pairs_by_depth, max_pairs_per_depth=5)

2026-02-27 17:34:35 src.goals.probe DEBUG    depth=1: 4 nodes, 1 positive, 0 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=2: 16 nodes, 1 positive, 0 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=3: 20 nodes, 2 positive, 1 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=4: 32 nodes, 3 positive, 2 negative


2026-02-27 17:34:35 src.goals.probe DEBUG    depth=5: 32 nodes, 3 positive, 3 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=6: 33 nodes, 3 positive, 3 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=7: 32 nodes, 3 positive, 3 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=8: 11 nodes, 3 positive, 4 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=9: 9 nodes, 3 positive, 4 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=10: 9 nodes, 3 positive, 4 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=11: 8 nodes, 3 positive, 4 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=12: 10 nodes, 3 positive, 4 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=13: 10 nodes, 4 positive, 4 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=14: 12 nodes, 5 positive, 4 negative
2026-02-27 17:34:35 src.goals.probe DEBUG    depth=15: 15 nodes, 6 positive, 4 negative
2026-02-27 17:34:35 src.goals.probe DEBU

Contrastive Pairs Summary

Total pairs: 167

Depths with pairs: 14

Depth range: (3, 16)

... (39 outputs left out)

... and 19 more pairs

Depth 16 (12 pairs)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Positive Path                                ┃ P(green) ┃ Negative Path                                ┃ P(red) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ '1].isupper() and s == s\[::-1] if s else    │     1.00 │ '1] and s\[0].isupper() and s == s\[::-1'    │   1.00 │
│ False'                                       │          │                                              │        │
│ '1].isupper() and s == s\[::-1] if s else    │     1.00 │ "0] != '' and s\[0].isupper() and s == s"    │   1.00 │
│ False'                                       │          │                                              │        │
│ '1].isupper() and s == s\[::-1] if s else    │     1.00 │ '].upper() == s and s\[0].isupper() and s    │   0.98 │
│ False'                                       │          │ =='                                          │        │
│ '1].isupper() and s == s\[::-1] and len(s)'  │     1.00 │ '1] and s\[0].isupper() and s == s\[::-1'    │   1.00 │
│ '1].isupper() and s == s\[::-1] and len(s)'  │     1.00 │ "0] != '' and s\[0].isupper() and s == s"    │   1.00 │
└──────────────────────────────────────────────┴──────────┴──────────────────────────────────────────────┴────────┘

... and 7 more pairs

In [17]:
# Inspect outcome probabilities at a specific depth
depth_to_inspect = 3
from src.goals.probe import collect_nodes_at_depth

nodes = collect_nodes_at_depth(root, depth_to_inspect)
print(f"Nodes at depth {depth_to_inspect}: {len(nodes)}")

for node in nodes[:10]:  # Show first 10
    if not node.is_leaf() and not node.is_terminal:
        probs = compute_outcome_probs(node)
        path_text = node.get_path_text()[:40]
        print(f"  '{path_text}...' -> {probs}")

Nodes at depth 3: 20
  '1].is...' -> OutcomeProbs(green=0.980, red=0.017, uncertain=0.004)
  '1] ==...' -> OutcomeProbs(green=0.140, red=0.611, uncertain=0.248)
  '1] >=...' -> OutcomeProbs(green=0.885, red=0.000, uncertain=0.115)
  '1] !=...' -> OutcomeProbs(green=0.463, red=0.000, uncertain=0.537)
  '1] and...' -> OutcomeProbs(green=0.000, red=0.744, uncertain=0.256)
  '0] !=...' -> OutcomeProbs(green=0.000, red=0.422, uncertain=0.578)


In [20]:
# Look at a specific contrastive pair in detail
if pairs_by_depth:
    # Get pairs from the first depth that has them
    first_depth = min(pairs_by_depth.keys())
    pairs = pairs_by_depth[first_depth]
    
    if pairs:
        pair = pairs[0]
        print(f"[bold]Example Contrastive Pair at depth {first_depth}:[/bold]")
        print()
        print(f"[green]Positive path:[/green] {repr(pair.positive.get_path_text())}")
        print(f"  Outcome probs: {pair.pos_probs}")
        print(f"  Node has {len(pair.positive.children)} children")
        print()
        print(f"[red]Negative path:[/red] {repr(pair.negative.get_path_text())}")
        print(f"  Outcome probs: {pair.neg_probs}")
        print(f"  Node has {len(pair.negative.children)} children")
else:
    print("[yellow]No contrastive pairs found. Try adjusting thresholds or running more MCTS iterations.[/yellow]")

[bold]Example Contrastive Pair at depth 3:[/bold]

[green]Positive path:[/green] '1].is'
  Outcome probs: OutcomeProbs(green=0.980, red=0.017, uncertain=0.004)
  Node has 4 children

[red]Negative path:[/red] '1] and'
  Outcome probs: OutcomeProbs(green=0.000, red=0.744, uncertain=0.256)
  Node has 4 children
